# 4a evaluation quick look (single year)

Interactive, single-year version of the Slurm evaluation in
[`input/slurm-scripts/4a_analysis_script.py`](../input/slurm-scripts/4a_analysis_script.py)
(manual mode with `--year-min == --year-max`). The helper functions below are the
same as in the script, so the numbers are directly comparable with a full 4a run.

**Usage:** point `ARGS_FILE` in the config cell at the args file of the reconstruction
run (e.g. `input/levante/eval-cnn-cerra-t2m-ger_sort_en4_po1_batch.txt`; a relative path
is tried against the notebook CWD and the repo root) and choose a `YEAR`
(`None` = newest test year), then *Run All*. Variable, NN output dir, reference
files, years and mask are then ALL resolved from that args file, exactly like
`4a_analysis_script.py --args-file` (if the file is missing, the notebook lists the
args files available in that directory):

- variable <- `--config-name` (fallback: `VAR` below)
- NN dir   <- `{evaluation-dirs}/{config-name}/{run-name}/data/` (a relative
  `--evaluation-dirs` such as `./evaluations/` is tried against the args-file dir
  first, then the notebook CWD / repo root — first existing directory wins, since
  `./evaluations` lives in the run directory, not next to the args file)
- reference <- the LAST comma-separated `--test-names` JSON (years from the file names;
  list files are searched relative to the args file first, then under the repo's
  `data/` tree: `data/test`, `data/train`, `data/val`, `data/masks`, ...)
- mask     <- first file of the first `--steady-masks` JSON (searched under
  `data/masks/` too), or the `--steady-mask-file` template
  (`{var}`/`{var_nc}`/`{reg}`/`{year}` tokens are substituted)

With `ARGS_FILE = ""` the manual fields in the config cell are used instead.
Requires the repo environment (`environment.yml`: xarray, dask, matplotlib, pandas).

For the selected year it computes:

- per-grid-cell **maps**: bias, MSE/RMSE, explained variance, correlation/R²,
  error quantiles, variance ratio, fraction of gap closed,
- **pooled scalar statistics** over all valid space-time points,
- **point/area statistics** (default: Hamburg) and the area-mean **time series**,
- **QQ plot**, value distribution / empirical CDF, and **ROC/AUC/PSS** for
  exceedance events at the reference quantiles.

Differences to the full script: figures are shown inline (and additionally saved to
`OUTPUT_DIR` if set), and the deseasonalization pass is skipped — with a single year
each day is its own climatology, so anomalies would be zero (the 4a script skips
`--deseasonalize` for a single year for the same reason).

In [ ]:
# ======================= EDIT ME =======================
ARGS_FILE = ""         # args file of the reconstruction run, e.g.
                       # "input/levante/eval-cnn-cerra-t2m-ger_sort_en4_po1_batch.txt"
                       # (relative: tried against the notebook CWD and the repo root;
                       #  paths inside the args file resolve against the args file itself).
                       # If set, VAR / paths / reference files / mask are ALL read from it.
YEAR = None            # the single year to look at; None = newest test year (args-file mode)

# ---- manual mode (only used when ARGS_FILE = "") ----
VAR = "t2m"            # variable/config name used in the file names
PATH_ORIGINAL = "/work/uo1075/u244085/cerra_data/t2m/"
PATH_NN = "/work/uo1075/u244085/trees_in_the_city/ML_downscaling/Physics-Informed-CNN-Reconstructions/evaluations/t2m/cerra-germany-0.05x0.05-sorted/data/"

FILE_PATTERN_NN = "cerra_dm_{var}_ger_{year}_crai_output.nc"   # NN output, relative to PATH_NN
FILE_PATTERN_REF = "cerra_dm_{var}_ger_{year}.nc"              # reference, relative to PATH_ORIGINAL

VAR_FILE = ""          # NetCDF variable in the reference file ("" = auto-detect, e.g. "2t")
VAR_FILE_NN = ""       # NetCDF variable in the NN output file ("" = auto-detect, e.g. "prediction")
RENAME_NN = ""         # optional NN coord renames, e.g. "longitude:lon,latitude:lat,valid_time:time"

MASK_FILE = ""         # e.g. ".../cerra_dm_t2m_ger_2016_steady_mask.nc"; "" = no mask
MASK_VAR = ""          # mask variable name; "" = auto (single data var in the file)
MASK_VALUE = 0         # mask value that defines the selection
MASK_MODE = "exclude"  # "exclude" = NaN cells where mask == MASK_VALUE; "keep" = keep only those

POINT_NAME = "Hamburg"           # point/area statistics location
POINT_LAT, POINT_LON = 53.5511, 9.9937
POINT_RADIUS_KM = 0              # >0: pool over ALL cells within this radius (km); 0 = nearest cell

QUANTILES = [0.05, 0.25, 0.5, 0.75, 0.95]   # per-cell error quantiles
ROC_QUANTILES = [0.5, 0.9, 0.95, 0.99]     # reference quantiles for the ROC/AUC/PSS analysis
QQ_STRIDE = 10               # time sub-sampling stride for the QQ / distribution samples
CHUNK_TIME = 30              # dask chunk size along time

OUTPUT_DIR = ""            # if set, figures are also saved there as <var>_<year>_*.png

# ---- color limits (vmin, vmax) for the map panels; None = auto ----
# auto: bias/error quantiles = symmetric around 0, variance_ratio = symmetric around 1,
# mse/rmse = data range. Set numbers to fix the limits, e.g. "bias": (-1.0, 1.0).
# Individual error-quantile maps can be overridden with "q05", "q25", ... keys.
COLOR_LIMITS = {
    "mse": (None, None),
    "bias": (None, None),
    "rmse": (None, None),
    "explained_variance": (-1, 1),
    "correlation": (-1, 1),
    "variance_ratio": (None, None),
    "fraction_gap_closed": (-1, 1),
    "error_quantiles": (None, None),
}
# =======================================================

## 1. Helpers

Same logic as `4a_analysis_script.py` (token substitution, variable auto-detection,
time-axis fix, ocean/mask handling, point mask, pooled statistics, ROC/AUC).

In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# variable name used in file names / config-name  ->  NetCDF variable name inside the files
VAR_FILE_DEFAULTS = {
    "t2m": "2t",
    "orography": "orog",
    "land_sea_mask": "lsm",
    "tp": "tp",
    "relative_humidity": "2r",
    "surface_pressure": "sp",
    "wind_speed": "10si",
    "wind_direction": "10wdir",
    "tciwv": "tciwv",
}
VAR_CANDIDATES = ["2t", "t2m", "tas", "air_temperature"]


def extract_year(path):
    """First 4-digit year (19xx/20xx) in the file name, else None."""
    m = re.findall(r"(?:19|20)\d{2}", Path(path).stem)
    return int(m[0]) if m else None


def subst_tokens(s, var, year=None):
    """Substitute {var}, {var_nc} and {year} in a file-name pattern by plain
    string replacement (stray braces cannot crash the run)."""
    s = str(s)
    s = s.replace("{var_nc}", VAR_FILE_DEFAULTS.get(str(var), str(var)))
    s = s.replace("{var}", str(var))
    if year is not None:
        s = s.replace("{year}", str(year))
    return s


def pattern_path(base_dir, pattern, var, year):
    """Join base_dir with the file-name pattern after token substitution."""
    return Path(base_dir) / subst_tokens(pattern, var, year)


def parse_args_file(path):
    """Parse an args file: one '--key value' pair per line. '#' lines are ignored;
    bare flags ('--normalize-data') are stored as True. Later occurrences win."""
    cfg = {}
    for raw in Path(path).read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        tokens = line.split()
        if not tokens[0].startswith("--"):
            continue
        key = tokens[0][2:]
        if len(tokens) > 1 and not tokens[1].startswith("--"):
            cfg[key] = tokens[1]
        else:
            cfg[key] = True
    return cfg


def read_data_list(list_path):
    """Read a data-list JSON (or plain text file) -> list of file paths. Supports a
    JSON array of strings, a JSON dict with 'files'/'data'/'paths' keys, or one path
    per line. Directories expand to their sorted *.nc contents. Relative paths
    resolve against the list file's location."""
    p = Path(list_path)
    if not p.is_file():
        raise FileNotFoundError(f"Data list not found: {p}")
    try:
        content = json.loads(p.read_text())
    except json.JSONDecodeError:
        entries = [l.strip() for l in p.read_text().splitlines()
                   if l.strip() and not l.strip().startswith("#")]
    else:
        if isinstance(content, str):
            content = [content]
        if isinstance(content, list):
            entries = [str(e) for e in content]
        elif isinstance(content, dict):
            for key in ("files", "data", "paths", "datasets"):
                if key in content:
                    entries = [str(e) for e in content[key]]
                    break
            else:
                raise ValueError(f"Unrecognised data-list format in {p} (keys: {list(content)})")
        else:
            raise ValueError(f"Unrecognised data-list format in {p}")
    files = []
    for e in entries:
        ep = Path(e)
        if not ep.is_absolute():
            ep = p.parent / ep
        ep = ep.resolve()
        if ep.is_dir():
            files.extend(sorted(str(f) for f in ep.glob("*.nc")))
        else:
            files.append(str(ep))
    return files


def find_args_file(args_file):
    """Resolve ARGS_FILE to an existing file. The string is first expanded with
    os.path.expanduser/expandvars (so \"~/...\" and \"$VAR/...\" work), then tried
    as given, relative to the notebook CWD, relative to its parent, and relative
    to the repo root inferred from the installed climatereconstructionai package.
    On failure, raises FileNotFoundError showing the CWD, every path tried, and the
    eval args files available near those locations."""
    raw = os.path.expandvars(os.path.expanduser(str(args_file)))
    cwd = Path.cwd()
    cands = []
    for base in (None, cwd, cwd.parent):
        p = Path(raw) if base is None else base / raw
        if p not in cands:
            cands.append(p)
    try:
        import climatereconstructionai as _pkg
        p = Path(_pkg.__file__).resolve().parents[1] / raw
        if p not in cands:
            cands.append(p)
    except Exception:
        pass
    for c in cands:
        if c.is_file():
            return c.resolve()
    hint = ""
    rel_parent = Path(raw).parent
    search = []
    for d in (rel_parent, cwd, cwd.parent, cwd.parent / rel_parent):
        d = d if d.is_absolute() else cwd / d
        search.append(d)
    for d in dict.fromkeys(search):
        if d.is_dir():
            avail = sorted(d.glob("eval-*.txt"))
            if avail:
                hint = (f"\nEval args files in {d} (first 10):\n  "
                        + "\n  ".join(str(c) for c in avail[:10]))
                break
    raise FileNotFoundError(
        f"Args file not found: {args_file!r} (expanded: {raw!r}, CWD: {cwd}).\n"
        f"Tried: {[str(c) for c in cands]}.{hint}\n"
        "Set ARGS_FILE to an existing args file (absolute path, or relative to the "
        "notebook CWD / repo root; ~ and $VAR are expanded), or \"\" for manual mode.")


def repo_root_candidates(extra=()):
    """Directories that may be the repo root: the notebook CWD and its parent, the
    given extra paths (e.g. the args-file dir) and their parents, and the parent of
    the installed climatereconstructionai package."""
    cwd = Path.cwd()
    roots = [cwd, cwd.parent]
    for e in extra:
        p = Path(e)
        roots.append(p)
        roots.extend(p.resolve().parents[:3])
    try:
        import climatereconstructionai as _pkg
        roots.append(Path(_pkg.__file__).resolve().parents[1])
    except Exception:
        pass
    return list(dict.fromkeys(roots))


def find_data_list_file(entry, base, split=None):
    """Locate a data-list JSON referenced from an args file (e.g. 't2m/cerra_t2m_ger_
    target_test.json' for --test-names). Tried in order: relative to the args file
    (as 4a_analysis_script.py does), then under each repo-root candidate, then under
    the repo's data/ tree, preferring data/<split>/ (e.g. 'test' for --test-names,
    'masks' for --steady-masks) and then any data/<split>/... subdirectory."""
    raw = os.path.expandvars(os.path.expanduser(str(entry)))
    p = Path(raw)
    cands = [p if p.is_absolute() else base / p]
    roots = repo_root_candidates(extra=(base,))
    for root in roots:
        cands.append(root / raw)
        data_dir = root / "data"
        if split:
            cands.append(data_dir / split / raw)
        if data_dir.is_dir():
            cands.extend(sorted(data_dir.glob(f"*/{raw}")))
    tried = []
    for c in dict.fromkeys(cands):
        tried.append(c)
        if c.is_file():
            return c.resolve()
    found = []
    var_dir = Path(raw).parent  # e.g. "t2m" for "t2m/cerra_..._test.json"
    for root in roots:
        d = root / "data"
        if not d.is_dir():
            continue
        if str(var_dir) not in ("", "."):
            found.extend(sorted(d.glob(f"*/{var_dir}/*.json")))
        else:
            found.extend(sorted(d.glob("*/*.json")))
    hint = ("\nAvailable list files under data/ (first 10):\n  "
            + "\n  ".join(str(f) for f in found[:10])) if found else ""
    raise FileNotFoundError(
        f"Data list not found: {entry!r} (args file dir: {base}).\n"
        f"Tried: {[str(c) for c in tried]}.{hint}")


def resolve_run_path(value, base, want_dir=False):
    """Resolve a relative path from an args file. Like crai-evaluate, relative paths
    belong to the RUN directory (repo root), while 4a_analysis_script.py resolves
    them against the args-file dir — so try the args-file dir first, then each
    repo-root candidate (notebook CWD, its parent, package root, args-file parents),
    and return the first existing one; if none exists, fall back to the args-file-dir
    based path so later checks can report it."""
    raw = os.path.expandvars(os.path.expanduser(str(value)))
    p = Path(raw)
    if p.is_absolute():
        return p
    roots = repo_root_candidates(extra=(base,))
    cands = list(dict.fromkeys([base / raw] + [root / raw for root in roots]))
    for c in cands:
        if (c.is_dir() if want_dir else c.is_file()):
            return c.resolve()
    return base / raw


def resolve_from_args_file(args_file, file_pattern_nn, var=None):
    """Config-file mode of 4a_analysis_script.py: extract variable, NN output dir,
    the {year: (nn_file, ref_file)} mapping, the mask and the config name from the
    args file. Returns (var, path_original, path_nn, files, mask_file, config_name,
    region); mask_file may still contain {var}/{var_nc}/{reg}/{year} tokens when it
    comes from a --steady-mask-file template."""
    cfg = parse_args_file(args_file)
    base = Path(args_file).resolve().parent

    def cfg_get(key, default=None):
        v = cfg.get(key, default)
        return v if v is not True else default

    var = var or cfg_get("config-name")
    if not var:
        raise ValueError("Neither VAR nor --config-name (in the args file) is set")

    run_name = cfg_get("run-name")
    if not run_name:
        raise ValueError("--run-name is missing in the args file")
    eval_dir = resolve_run_path(
        str(cfg_get("evaluation-dirs", "evaluations")).split(",")[0].strip(),
        base, want_dir=True)
    config_name = cfg_get("config-name", var)
    path_nn = eval_dir / config_name / run_name / "data"

    test_names = cfg_get("test-names")
    if not test_names:
        raise ValueError("--test-names is missing in the args file")
    target_json = [t for t in str(test_names).split(",") if t.strip()][-1].strip()
    target_p = find_data_list_file(target_json, base, split="test")
    print(f"test list : {target_p}")
    ref_files = read_data_list(target_p)

    files = {}
    for rf in ref_files:
        y = extract_year(rf)
        if y is None:
            print(f"WARNING: no year in the file name -> skipped: {rf}")
            continue
        files[y] = (pattern_path(path_nn, file_pattern_nn, var, y), Path(rf))
    if not files:
        raise ValueError("No reference files with a year in their name")

    path_original = str(Path(ref_files[0]).parent)

    mask_file = None
    steady = cfg_get("steady-masks")
    if steady:
        steady_json = [s for s in str(steady).split(",") if s.strip()][0].strip()
        sp = find_data_list_file(steady_json, base, split="masks")
        print(f"mask list : {sp}")
        mask_list = read_data_list(sp)
        mask_file = mask_list[0] if mask_list else None
    else:
        mask_file = cfg_get("steady-mask-file")  # template with {var}/{reg}/{year} tokens
    region = cfg_get("region", "ger")
    return var, path_original, str(path_nn), files, mask_file, config_name, region


def check_files(files):
    """Fail fast with a clear message if an input file is missing; for the first
    missing file, list what IS in its directory to help fix the pattern."""
    missing = sorted({f for f in files if not Path(f).is_file()})
    if missing:
        msg = "Missing input files:\n  " + "\n  ".join(str(m) for m in missing)
        parent = Path(missing[0]).parent
        if parent.is_dir():
            present = sorted(f.name for f in parent.iterdir() if f.is_file())
            if present:
                msg += ("\n\nFiles present in %s (first 5):\n  " % parent
                        + "\n  ".join(present[:5])
                        + "\n  ... adjust FILE_PATTERN_NN / FILE_PATTERN_REF if these do not match.")
        raise FileNotFoundError(msg)


def detect_spatial_names(ds):
    lat = next((d for d in ("lat", "latitude", "y") if d in ds.dims or d in ds.coords), None)
    lon = next((d for d in ("lon", "longitude", "x") if d in ds.dims or d in ds.coords), None)
    if lat is None or lon is None:
        raise ValueError(f"Could not detect the spatial dims (dims: {list(ds.dims)}, "
                         f"coords: {list(ds.coords)}).")
    return lat, lon


def detect_var_file(var, ref_file, var_file=None):
    """Auto-detect the NetCDF variable name from the reference file."""
    if var_file:
        return var_file
    candidates = [VAR_FILE_DEFAULTS.get(var), var] + VAR_CANDIDATES
    candidates = [c for c in candidates if c]
    with xr.open_dataset(str(ref_file)) as ds:
        hit = [v for v in candidates if v in ds.data_vars]
        if not hit:
            raise ValueError(f"None of {candidates} found in {ref_file}.\n"
                             f"Available variables: {list(ds.data_vars)}. Set VAR_FILE explicitly.")
    return hit[0]


def detect_var_file_nn(var, nn_file, var_file, var_file_nn=None):
    """NetCDF variable name in the NN output file (may differ from the reference,
    e.g. 'prediction' instead of '2t')."""
    if var_file_nn:
        return var_file_nn
    with xr.open_dataset(nn_file) as ds:
        if var_file in ds.data_vars:
            return var_file
        if len(ds.data_vars) == 1:
            name = list(ds.data_vars)[0]
            print(f"NN variable: '{name}' (auto-detected; the reference uses '{var_file}')")
            return name
        candidates = [VAR_FILE_DEFAULTS.get(var), var] + VAR_CANDIDATES
        hit = [c for c in candidates if c and c in ds.data_vars]
        if hit:
            print(f"NN variable: '{hit[0]}' (auto-detected; the reference uses '{var_file}')")
            return hit[0]
        raise ValueError(f"Cannot determine the data variable in the NN file\n  {nn_file}\n"
                         f"(the reference variable '{var_file}' is absent; available: "
                         f"{list(ds.data_vars)}). Set VAR_FILE_NN explicitly.")


def ensure_datetime_time(da, file_ref):
    """Make sure the 'time' axis is a decoded datetime axis; rebuild it as a full
    calendar year starting Jan 1st when it is stored as plain numbers."""
    t = da["time"]
    if np.issubdtype(t.dtype, np.datetime64) or t.dtype == object:
        return da
    year = extract_year(file_ref)
    n = int(t.size)
    steps_per_day = None
    if year is not None:
        for days in (365, 366):
            if n % days == 0:
                steps_per_day = n // days
                break
    if steps_per_day is None:
        raise ValueError(f"The 'time' axis of\n  {file_ref}\nis not decoded (dtype {t.dtype}, "
                         f"{n} steps) and cannot be rebuilt automatically. Fix the time units.")
    freq = pd.Timedelta(hours=24 / steps_per_day)
    new_time = pd.date_range(f"{year}-01-01 00:00", periods=n, freq=freq)
    print(f"WARNING: numeric 'time' axis in {Path(file_ref).name} (dtype {t.dtype}) -> "
          f"rebuilt as {n} steps of {freq} starting {year}-01-01.")
    return da.assign_coords(time=new_time)


def load_pair(file_nn, file_ref, chunks, var_file, var_file_nn=None,
              keep_mask=None, rename_nn=""):
    """Open NN + reference, fix the NN time axis, mask ocean cells (reference NaN
    pattern), apply the cell mask, align. Returns (nn, ref) with identical axes."""
    ref = xr.open_dataset(file_ref, chunks=chunks)[var_file]
    ref = ensure_datetime_time(ref, file_ref)

    # open the NN file WITHOUT chunks first: chunk keys are only valid AFTER the renames
    nn_ds = xr.open_dataset(file_nn)
    if rename_nn:
        renames = dict(pair.split(":") for pair in rename_nn.split(",") if ":" in pair)
        renames = {k: v for k, v in renames.items() if k in nn_ds.dims or k in nn_ds.coords}
        if renames:
            nn_ds = nn_ds.rename(renames)

    nn = nn_ds[var_file_nn or var_file]
    nn = nn.chunk({k: v for k, v in chunks.items() if k in nn.dims})

    if "time" not in nn.dims:
        raise ValueError(f"NN file\n  {file_nn}\nhas no 'time' dimension after the renames "
                         f"(dims: {list(nn.dims)}). Check RENAME_NN.")

    # the NN output time axis is wrong (constant year); overwrite with the reference axis
    n_ref = ref.sizes["time"]
    n_nn = nn.sizes["time"]
    if n_nn > n_ref:
        nn = nn.isel(time=slice(n_ref))
    elif n_nn < n_ref:
        raise ValueError(f"NN file\n  {file_nn}\nhas {n_nn} timesteps, but the reference\n"
                         f"  {file_ref}\nhas {n_ref}. Cannot align the time axes.")
    nn["time"] = ref["time"]

    # mask ocean / invalid cells using the reference NaN pattern
    nn = xr.where(np.isnan(ref), np.nan, nn)
    nn, ref = xr.align(nn, ref, join="exact")

    if keep_mask is not None:
        nn = nn.where(keep_mask)
        ref = ref.where(keep_mask)
    return nn, ref


def build_keep_mask(mask_file, mask_var, mask_value, mask_mode, ref_file):
    """Load the mask file and return a (lat, lon) boolean DataArray of the cells to
    KEEP, aligned to the data grid."""
    if not Path(mask_file).is_file():
        raise FileNotFoundError(f"Mask file not found: {mask_file}")

    with xr.open_dataset(mask_file) as mds:
        if mask_var:
            if mask_var not in mds.variables:
                raise ValueError(f"Mask variable '{mask_var}' not found in {mask_file}. "
                                 f"Available: {list(mds.data_vars)}.")
            m = mds[mask_var]
        elif len(mds.data_vars) == 1:
            m = mds[list(mds.data_vars)[0]]
        else:
            raise ValueError(f"Mask file {mask_file} contains several variables "
                             f"({list(mds.data_vars)}). Set MASK_VAR explicitly.")
        m = m.squeeze().load()

    m_lat, m_lon = detect_spatial_names(m)
    with xr.open_dataset(ref_file) as ds0:
        d_lat, d_lon = detect_spatial_names(ds0)
        d_lat_vals = ds0[d_lat].values
        d_lon_vals = ds0[d_lon].values

    if m_lat != d_lat:
        m = m.rename({m_lat: d_lat})
    if m_lon != d_lon:
        m = m.rename({m_lon: d_lon})

    same_grid = (m[d_lat].size == d_lat_vals.size
                 and m[d_lon].size == d_lon_vals.size
                 and np.allclose(m[d_lat].values, d_lat_vals)
                 and np.allclose(m[d_lon].values, d_lon_vals))
    if not same_grid:
        print("WARNING: mask grid differs from the data grid -> nearest-neighbour reindexing")
        m = m.reindex({d_lat: d_lat_vals, d_lon: d_lon_vals}, method="nearest")

    keep = (m == mask_value) if mask_mode == "keep" else (m != mask_value)

    extra_dims = sorted(set(keep.dims) - {d_lat, d_lon})
    for extra in extra_dims:
        keep = keep.all(extra)
    keep = keep.squeeze().reset_coords(drop=True)
    n_keep = int(keep.sum())
    print(f"Mask: mode={mask_mode}, value={mask_value} -> {n_keep}/{keep.size} cells kept "
          f"({100.0 * n_keep / max(keep.size, 1):.1f}%)")
    return keep


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (degrees in, numpy arrays allowed)."""
    r = 6371.0
    lat1, lon1, lat2, lon2 = (np.radians(lat1), np.radians(lon1),
                              np.radians(lat2), np.radians(lon2))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


def build_point_mask(ds, lat_name, lon_name, lat, lon, radius_km):
    """Boolean (lat, lon) DataArray of the cells used for the point/area statistics:
    ALL cells within radius_km, or only the nearest cell when radius_km == 0."""
    lat_vals = ds[lat_name].values
    lon_vals = ds[lon_name].values
    if radius_km and radius_km > 0:
        la2, lo2 = np.meshgrid(lat_vals, lon_vals, indexing="ij")
        mask = haversine_km(lat, lon, la2, lo2) <= radius_km
    else:
        mask = np.zeros((lat_vals.size, lon_vals.size), dtype=bool)
        mask[int(np.argmin(np.abs(lat_vals - lat))), int(np.argmin(np.abs(lon_vals - lon)))] = True
    return xr.DataArray(mask, dims=(lat_name, lon_name),
                        coords={lat_name: lat_vals, lon_name: lon_vals})


def pooled_stats(se, sse, sae, sr, ssr, sn, ssn, cnr, n):
    """Statistics pooled over n space-time points, computed from accumulated sums.
    NaN when n < 2."""
    out = {"n": int(n), "bias": np.nan, "rmse": np.nan, "mae": np.nan,
           "ev": np.nan, "corr": np.nan, "r2": np.nan, "var_ratio": np.nan}
    if n < 2:
        return out
    var_e = (sse - se * se / n) / (n - 1)
    var_r = (ssr - sr * sr / n) / (n - 1)
    var_n = (ssn - sn * sn / n) / (n - 1)
    cov_nr = (cnr - sn * sr / n) / (n - 1)
    corr = float(cov_nr / np.sqrt(var_n * var_r)) if var_n > 0 and var_r > 0 else np.nan
    out["bias"] = float(se / n)
    out["rmse"] = float(np.sqrt(sse / n))
    out["mae"] = float(sae / n)
    out["ev"] = float(1 - var_e / var_r) if var_r > 0 else np.nan
    out["corr"] = corr
    out["r2"] = corr ** 2 if np.isfinite(corr) else np.nan
    out["var_ratio"] = float(var_n / var_r) if var_r > 0 else np.nan
    return out


def roc_analysis(scores, events, threshold, max_points=200000):
    """ROC analysis for the binary `events` with continuous `scores`: AUC (rank /
    Mann-Whitney formula, ties averaged) and ROC curve points on a sub-sample of at
    most max_points points, plus the Peirce skill score PSS = TPR + TNR - 1 at
    `threshold`. Returns None for degenerate events."""
    scores = np.asarray(scores, dtype="float64").ravel()
    events = np.asarray(events, dtype=bool).ravel()
    finite = np.isfinite(scores)
    scores, events = scores[finite], events[finite]
    n = scores.size
    n_pos = int(events.sum())
    n_neg = n - n_pos
    if n < 2 or n_pos == 0 or n_neg == 0:
        return None
    tp = int(np.sum(events & (scores > threshold)))
    fp = int(np.sum(~events & (scores > threshold)))
    fn = n_pos - tp
    tn = n_neg - fp
    pss = (tp / (tp + fn) + tn / (tn + fp) - 1.0
           if (tp + fn) > 0 and (tn + fp) > 0 else np.nan)
    if n > max_points:
        idx = np.unique(np.linspace(0, n - 1, max_points).astype(int))
        s_sub, e_sub = scores[idx], events[idx]
    else:
        s_sub, e_sub = scores, events
    s_sorted = np.sort(s_sub)
    e_sorted = e_sub[np.argsort(s_sub, kind="mergesort")].astype(np.int64)
    p_sub = int(e_sorted.sum())
    if p_sub == 0 or p_sub == s_sorted.size:
        s_sorted = np.sort(scores)
        e_sorted = events[np.argsort(scores, kind="mergesort")].astype(np.int64)
        p_sub = n_pos
    uniq, inv, counts = np.unique(s_sorted, return_inverse=True, return_counts=True)
    starts = np.cumsum(counts) - counts
    avg_rank = starts + (counts + 1) / 2.0
    ranks = avg_rank[inv]
    auc = float((ranks[e_sorted == 1].sum() - p_sub * (p_sub + 1) / 2)
                / (p_sub * (s_sorted.size - p_sub)))
    thr = np.quantile(s_sub, np.linspace(0, 1, 201))
    cum_pos = np.concatenate([[0], np.cumsum(e_sorted)])
    k = np.searchsorted(s_sorted, thr, side="right")
    tp_gt = p_sub - cum_pos[k]
    fp_gt = (s_sorted.size - k) - tp_gt
    fpr = np.concatenate([[0.0], fp_gt / (s_sorted.size - p_sub), [1.0]])
    tpr = np.concatenate([[0.0], tp_gt / p_sub, [1.0]])
    return {"threshold": float(threshold), "auc": auc, "pss": float(pss),
            "fpr": fpr, "tpr": tpr}


## 2. Resolve the input files for the selected year

Same defaults as `4a_submit_evaluation.sh` (manual mode): reference
files and mask from the args file; with `ARGS_FILE = ""` the manual patterns
`cerra_dm_{var}_ger_{year}.nc` (reference) and `cerra_dm_{var}_ger_{year}_crai_output.nc`
(NN output) are used.

In [ ]:
if ARGS_FILE:
    args_file = find_args_file(ARGS_FILE)
    var, path_original, path_nn, files, mask_file, config_name, region = resolve_from_args_file(
        args_file, FILE_PATTERN_NN, var=VAR or None)
    years_avail = sorted(files)
    print(f"args file : {args_file}")
    print(f"NN dir    : {path_nn}")
    print(f"reference : {len(files)} test year(s) from the LAST --test-names list: "
          f"{years_avail[0]}-{years_avail[-1]}")
    year = YEAR if YEAR is not None else years_avail[-1]
    if year not in files:
        raise ValueError(f"YEAR={year} is not among the test years {years_avail}")
    file_nn, file_ref = files[year]
    VAR = var
    if mask_file and "{" in str(mask_file):
        mask_file = (str(mask_file)
                     .replace("{var_nc}", VAR_FILE_DEFAULTS.get(var, var))
                     .replace("{var}", var)
                     .replace("{reg}", region)
                     .replace("{year}", str(year)))
    if mask_file:
        mask_file = str(resolve_run_path(mask_file, Path(args_file).parent))
        print(f"mask file : {mask_file}")
else:
    config_name = None
    mask_file = MASK_FILE
    if YEAR is None:
        raise ValueError("Manual mode needs an explicit YEAR (or set ARGS_FILE)")
    year = YEAR
    file_nn = Path(PATH_NN) / subst_tokens(FILE_PATTERN_NN, VAR, year)
    file_ref = Path(PATH_ORIGINAL) / subst_tokens(FILE_PATTERN_REF, VAR, year)

check_files([file_nn, file_ref])

var_file = detect_var_file(VAR, file_ref, VAR_FILE or None)
var_file_nn = detect_var_file_nn(VAR, file_nn, var_file, VAR_FILE_NN or None)

with xr.open_dataset(file_ref) as ds0:
    unit = str(ds0[var_file].attrs.get("units", "")).strip()
    lat_name, lon_name = detect_spatial_names(ds0)
    point_mask = build_point_mask(ds0, lat_name, lon_name, POINT_LAT, POINT_LON, POINT_RADIUS_KM)

keep_mask = None
if mask_file:
    keep_mask = build_keep_mask(mask_file, MASK_VAR or None, MASK_VALUE, MASK_MODE, file_ref)

tag = f"{VAR}_{year}"
print(f"tag       : {tag}")
print(f"year      : {year}")
print(f"reference : {file_ref}")
print(f"NN output : {file_nn}")
print(f"variable  : {var_file} (NN file: {var_file_nn})" + (f" [{unit}]" if unit else ""))
print(f"config    : {config_name or 'manual mode'}")
print(f"point     : '{POINT_NAME}' -> {int(point_mask.sum())} grid cell(s)")

## 3. Load NN + reference and compute the per-cell statistics

One lazy pass over the year (like pass 1 of the script): per-cell maps plus pooled
moments and point/area moments are computed in a single `.compute()`.

In [ ]:
chunks = {"time": CHUNK_TIME}
nn, ref = load_pair(file_nn, file_ref, chunks, var_file, var_file_nn,
                    keep_mask=keep_mask, rename_nn=RENAME_NN)
err = nn - ref

nn_anom = nn - nn.mean("time")
ref_anom = ref - ref.mean("time")
ref_var = ref.var("time")
mad_ref = np.abs(ref_anom).mean("time")

members = {
    "bias": err.mean("time"),
    "mse": (err ** 2).mean("time"),
    "ev": xr.where(ref_var > 1e-12, 1 - err.var("time") / ref_var, np.nan),
    "corr": (nn_anom * ref_anom).mean("time") / (nn_anom.std("time") * ref_anom.std("time")),
    "var_ratio": xr.where(ref_var > 1e-12, nn.var("time") / ref_var, np.nan),
    "fgc": xr.where(mad_ref > 1e-12, 1 - np.abs(err).mean("time") / mad_ref, np.nan),
    "se": err.sum(),
    "sse": (err ** 2).sum(),
    "cnt": err.count(),
    "nn_sub": nn.isel(time=slice(None, None, QQ_STRIDE)),
    "ref_sub": ref.isel(time=slice(None, None, QQ_STRIDE)),
}
for q in QUANTILES:
    qm = err.quantile(q, dim="time")
    if "quantile" in qm.coords:
        qm = qm.drop_vars("quantile")
    members[f"q{int(q * 100)}"] = qm

# point/area pooled moments in the same pass
err_r = err.where(point_mask)
ref_r = ref.where(point_mask)
nn_r = nn.where(point_mask)
members.update({
    "pt_se": err_r.sum(), "pt_sse": (err_r ** 2).sum(), "pt_sae": np.abs(err_r).sum(),
    "pt_sr": ref_r.sum(), "pt_ssr": (ref_r ** 2).sum(),
    "pt_sn": nn_r.sum(), "pt_ssn": (nn_r ** 2).sum(), "pt_cnr": (nn_r * ref_r).sum(),
    "pt_cnt": err_r.count(),
    "pt_nn_mean": nn_r.mean(dim=[lat_name, lon_name]),
    "pt_ref_mean": ref_r.mean(dim=[lat_name, lon_name]),
})

stats = xr.Dataset(members).compute()

bias_map = stats["bias"]
mse_map = stats["mse"]
rmse_map = np.sqrt(mse_map)
ev_map = stats["ev"]
corr_map = stats["corr"]
r2_map = corr_map ** 2
var_ratio_map = stats["var_ratio"]
fgc_map = stats["fgc"]
q_maps = {q: stats[f"q{int(q * 100)}"] for q in QUANTILES}

print(f"time steps : {ref.sizes['time']}   grid: {lat_name}={ref.sizes[lat_name]}, {lon_name}={ref.sizes[lon_name]}")
print(f"valid space-time points: {int(stats['cnt'])}")

## 4. Scalar and point/area statistics

Pooled over all valid space-time points (domain) and over the point/area cells
(`pooled_stats`, same accumulated-sums formulas as the script).

In [ ]:
n_total = int(stats["cnt"])
bias_scalar = float(stats["se"]) / n_total
rmse_scalar = float(np.sqrt(float(stats["sse"]) / n_total))

nn_all = stats["nn_sub"].values.ravel().astype("float32")
ref_all = stats["ref_sub"].values.ravel().astype("float32")
finite = np.isfinite(nn_all) & np.isfinite(ref_all)
nn_all, ref_all = nn_all[finite], ref_all[finite]

mae_scalar = float(np.mean(np.abs(nn_all - ref_all)))
mad_ref_scalar = float(np.mean(np.abs(ref_all - ref_all.mean())))
corr_scalar = float(np.corrcoef(ref_all, nn_all)[0, 1])
scalar = {
    "bias": bias_scalar,
    "RMSE": rmse_scalar,
    "MAE": mae_scalar,
    "EV": float(1 - np.var(nn_all - ref_all) / np.var(ref_all)),
    "corr": corr_scalar,
    "R2": corr_scalar ** 2,
    "var_ratio": float(np.var(nn_all) / np.var(ref_all)),
    "frac_gap_closed": float(1 - mae_scalar / mad_ref_scalar),
    "n": n_total,
}

point = pooled_stats(float(stats["pt_se"]), float(stats["pt_sse"]), float(stats["pt_sae"]),
                     float(stats["pt_sr"]), float(stats["pt_ssr"]), float(stats["pt_sn"]),
                     float(stats["pt_ssn"]), float(stats["pt_cnr"]), int(stats["pt_cnt"]))

pd.DataFrame({
    "domain (all valid cells)": scalar,
    f"{POINT_NAME} ({int(point_mask.sum())} cell(s))": point,
}).T

## 5. Maps (per grid cell, map + histogram)

Same figures as the 4a script (`map_and_hist`): shown inline and, if `OUTPUT_DIR` is
set, saved as `<var>_<year>_<name>.png`.

Color limits of the map panels are set in `COLOR_LIMITS` (config cell);
`None` = automatic (symmetric around 0 for bias/quantiles, around 1 for the
variance ratio, data range otherwise).

In [ ]:
OUT = Path(OUTPUT_DIR) if OUTPUT_DIR else None
if OUT is not None:
    OUT.mkdir(parents=True, exist_ok=True)

meta_line = (f"{VAR} ({var_file})" + (f" [{unit}]" if unit else "") + f", {year}"
             + f", config: {config_name or 'manual mode'}")


def clims(fname, auto):
    """(vmin, vmax) for a map panel: the COLOR_LIMITS entry if set, else the auto values."""
    vmin, vmax = COLOR_LIMITS.get(fname, (None, None))
    return (auto[0] if vmin is None else vmin, auto[1] if vmax is None else vmax)


def show(fig, fname):
    if OUT is not None:
        fig.savefig(OUT / f"{tag}_{fname}.png", dpi=150)
    plt.show()
    plt.close(fig)


def map_and_hist(da, cmap, vmin, vmax, xlabel, unit_label, fname, title,
                 mean_line=True, extra_vlines=(), hist_title=None):
    fig, (ax_map, ax_hist) = plt.subplots(
        1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]}
    )
    da.plot(ax=ax_map, cmap=cmap, vmin=vmin, vmax=vmax, cbar_kwargs={"label": unit_label})
    ax_map.set_title(f"{title}\n{meta_line}")
    vals = da.values.ravel()
    vals = vals[np.isfinite(vals)]
    if len(vals):
        ax_hist.hist(vals, bins=51, color="C0", alpha=0.8)
        if mean_line:
            ax_hist.axvline(0, color="k", lw=1)
            ax_hist.axvline(vals.mean(), color="C3", ls="--",
                            label=f"mean = {vals.mean():.4f}\nmedian = {np.median(vals):.4f}")
        for x, kw in extra_vlines:
            ax_hist.axvline(x, **kw)
    ax_hist.set_yscale("log")
    ax_hist.set_xlabel(xlabel)
    ax_hist.set_ylabel("Number of grid cells")
    if hist_title:
        ax_hist.set_title(hist_title)
    if ax_hist.get_legend_handles_labels()[1]:
        ax_hist.legend(frameon=False)
    plt.tight_layout()
    show(fig, fname)


map_and_hist(mse_map, "viridis", *clims("mse", (None, None)), "MSE", "MSE", "mse",
             f"Mean MSE per grid cell, {year}",
             mean_line=False, hist_title="Distribution of per-cell MSE")

vmax = float(np.nanmax(np.abs(bias_map.values)))
map_and_hist(bias_map, "RdBu_r", *clims("bias", (-vmax, vmax)),
             "Bias (NN - reference)", "Bias (NN - reference)", "bias", f"Mean bias, {year}")

rmse_vals = rmse_map.values.ravel()
rmse_vals = rmse_vals[np.isfinite(rmse_vals)]
global_rmse = float(np.sqrt(np.mean(rmse_vals ** 2)))
map_and_hist(rmse_map, "viridis", *clims("rmse", (None, None)), "RMSE", "RMSE", "rmse",
             f"RMSE per grid cell, {year}", mean_line=False,
             extra_vlines=[(global_rmse, {"color": "C3", "ls": "--", "lw": 1,
                                          "label": f"global RMSE = {global_rmse:.4f}"})],
             hist_title="Distribution of per-cell RMSE")

map_and_hist(ev_map, "RdBu_r", *clims("explained_variance", (-1, 1)),
             "Explained variance", "Explained variance", "explained_variance",
             f"Explained variance per grid cell, {year}")
map_and_hist(corr_map, "RdBu_r", *clims("correlation", (-1, 1)),
             "Correlation", "Correlation", "correlation",
             f"Correlation per grid cell, {year}")

vr_vals = var_ratio_map.values.ravel()
vr_vals = vr_vals[np.isfinite(vr_vals)]
dev = min(max(float(np.max(np.abs(vr_vals - 1))) if vr_vals.size else 1.0, 0.5), 2.0)
map_and_hist(var_ratio_map, "RdBu_r", *clims("variance_ratio", (1 - dev, 1 + dev)),
             "Variance ratio (NN/ref)", "Variance ratio (NN/ref)", "variance_ratio",
             f"Variance ratio per grid cell, {year}")
map_and_hist(fgc_map, "RdBu_r", *clims("fraction_gap_closed", (-1, 1)),
             "Fraction of gap closed", "Fraction of gap closed", "fraction_gap_closed",
             f"Fraction of gap closed per grid cell, {year}")

### Error quantiles per grid cell

In [ ]:
vmax_q = max(float(np.nanmax(np.abs(m.values))) for m in q_maps.values())
q_lims = clims("error_quantiles", (-vmax_q, vmax_q))
fig, ax = plt.subplots(figsize=(8, 5))
for i, q in enumerate(QUANTILES):
    vals = q_maps[q].values.ravel()
    vals = vals[np.isfinite(vals)]
    ax.hist(vals, bins=51, range=q_lims, histtype="step", lw=1.5,
            color=f"C{i}", label=f"q={q:.2f}")
ax.axvline(0, color="k", lw=1)
ax.set_yscale("log")
ax.set_xlabel("Error quantile value (NN - reference)")
ax.set_ylabel("Number of grid cells")
ax.set_title(f"Per-cell error quantiles, {year}\n{meta_line}")
ax.legend(frameon=False)
plt.tight_layout()
show(fig, "error_quantiles_hist")

for q in QUANTILES:
    qm = q_maps[q]
    qq = clims(f"q{int(q * 100):02d}", q_lims)
    fig, (ax_map, ax_hist) = plt.subplots(
        1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]}
    )
    qm.plot(ax=ax_map, cmap="RdBu_r", vmin=qq[0], vmax=qq[1],
            cbar_kwargs={"label": f"q{int(q * 100):02d} of error (NN - reference)"})
    ax_map.set_title(f"Error quantile q={q:.2f} per grid cell, {year}\n{meta_line}")
    vals = qm.values.ravel()
    vals = vals[np.isfinite(vals)]
    ax_hist.hist(vals, bins=51, color="C0", alpha=0.8)
    ax_hist.axvline(0, color="k", lw=1)
    ax_hist.axvline(np.median(vals), color="C3", ls="--", lw=1,
                    label=f"median = {np.median(vals):.4f}")
    ax_hist.set_xlim(*qq)
    ax_hist.set_yscale("log")
    ax_hist.set_xlabel(f"q{int(q * 100):02d} of error (NN - reference)")
    ax_hist.set_ylabel("Number of grid cells")
    ax_hist.legend(frameon=False)
    plt.tight_layout()
    show(fig, f"q{int(q * 100):02d}")

## 6. Point time series, value distributions, QQ plot

In [ ]:
# point/area mean time series (e.g. Hamburg)
nn_pt = stats["pt_nn_mean"]
ref_pt = stats["pt_ref_mean"]
time_pt = nn_pt["time"]
fig, (ax_ts, ax_err) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax_ts.plot(time_pt, ref_pt, color="C1", lw=0.8, alpha=0.6, label="Reference")
ax_ts.plot(time_pt, nn_pt, color="C0", lw=0.8, alpha=0.6, label="NN")
ax_ts.set_ylabel(var_file)
ax_ts.set_title(f"{POINT_NAME} ({int(point_mask.sum())} cell(s), area mean), {year}\n{meta_line}")
ax_ts.legend(frameon=False)
err_pt = nn_pt - ref_pt
ax_err.plot(time_pt, err_pt, color="C3", lw=0.8, alpha=0.7)
ax_err.axhline(0, color="k", lw=1)
ax_err.set_ylabel("Error (NN - reference, area mean)")
ax_err.set_xlabel("time")
plt.tight_layout()
show(fig, f"point_{POINT_NAME.lower().replace(' ', '_')}_timeseries")

# distribution of values: NN vs reference (pooled, stride-sub-sampled samples)
fig, (ax_hist, ax_cdf) = plt.subplots(1, 2, figsize=(13, 5))
lo = min(float(nn_all.min()), float(ref_all.min()))
hi = max(float(nn_all.max()), float(ref_all.max()))
bins = np.linspace(lo, hi, 51)
ax_hist.hist(ref_all, bins=bins, density=True, alpha=0.6, color="C1", label="Reference")
ax_hist.hist(nn_all, bins=bins, density=True, alpha=0.6, color="C0", label="NN")
ax_hist.axvline(float(ref_all.mean()), color="C1", ls="--", lw=1,
                label=f"ref mean = {float(ref_all.mean()):.4f}")
ax_hist.axvline(float(nn_all.mean()), color="C0", ls="--", lw=1,
                label=f"NN mean  = {float(nn_all.mean()):.4f}")
ax_hist.set_xlabel(f"{var_file} value")
ax_hist.set_ylabel("Density")
ax_hist.set_title(f"Distribution of values, {year}\n{meta_line}")
ax_hist.legend(frameon=False)
for data, color, lab in ((ref_all, "C1", "Reference"), (nn_all, "C0", "NN")):
    xs = np.sort(data)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    ax_cdf.plot(xs, ys, color=color, lw=1.5, label=lab)
ax_cdf.set_xlabel(f"{var_file} value")
ax_cdf.set_ylabel("Cumulative fraction")
ax_cdf.set_title(f"Empirical CDF, {year}\n{meta_line}")
ax_cdf.legend(frameon=False)
plt.tight_layout()
show(fig, "value_distribution")

# pooled QQ plot
fig, ax = plt.subplots(figsize=(6, 6))
nn_sorted = np.sort(nn_all)
ref_sorted = np.sort(ref_all)
n_pts = min(20000, len(nn_sorted))
idx = np.unique(np.linspace(0, len(nn_sorted) - 1, n_pts).astype(int))
lims = [min(ref_sorted.min(), nn_sorted.min()), max(ref_sorted.max(), nn_sorted.max())]
ax.plot(lims, lims, "k--", lw=1, label="1:1")
ax.plot(ref_sorted[idx], nn_sorted[idx], ".", ms=2, color="C0", alpha=0.5)
ax.set_xlabel("Reference quantiles")
ax.set_ylabel("NN quantiles")
ax.set_title(f"Pooled QQ plot, {year}\n{meta_line}")
ax.legend(frameon=False)
plt.tight_layout()
show(fig, "qq")

## 7. ROC / AUC / PSS for exceedance events

Exceedance of the reference quantiles `ROC_QUANTILES` (pooled sub-sampled values,
same `roc_analysis` as the 4a script).

In [ ]:
roc_results = []
for q in ROC_QUANTILES:
    thr = float(np.quantile(ref_all, q))
    r = roc_analysis(nn_all, ref_all > thr, thr)
    if r is None:
        print(f"ROC q={q:.2f}: degenerate event, skipped")
        continue
    r["q"] = q
    roc_results.append(r)

if roc_results:
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="no skill")
    for i, r in enumerate(roc_results):
        ax.plot(r["fpr"], r["tpr"], lw=1.5, color=f"C{i}",
                label=f"q={r['q']:.2f}: AUC={r['auc']:.3f}, PSS={r['pss']:.3f}")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f"ROC curves for exceedance events, {year}\n{meta_line}")
    ax.legend(frameon=False, fontsize=8, loc="lower right")
    plt.tight_layout()
    show(fig, "roc")

    pd.DataFrame([{"q": r["q"], "threshold": r["threshold"],
                   "AUC": r["auc"], "PSS": r["pss"]} for r in roc_results])